In [ ]:
from platform import python_version
print(python_version())

### Cluster with Tahoe

### Huggingface: tahoebio/Tahoe-x1-embeddings

https://github.com/tahoebio/tahoe-x1

Tahoe-x1: Scaling Perturbation-Trained Single-Cell Foundation Models to 3 Billion Parameters


#### Memory

That's not a general "64 GB isn't enough" — swap is fully exhausted at 2.0G, which means something asked for tens of GB in one allocation. 

Given where you are in the pipeline, the culprit is almost certainly load_tahoe_de, and the arithmetic says so:

The DE table is ~4.09e9 rows over ~75k conditions × ~54k genes. 

Filtering to pancreas doesn't help much — roughly 
- 6 lines × 379 drugs × ~4 doses × 54k genes ≈ 5e8 rows, 
- materialised in pandas with gene/drug/cell_line_id as object-dtype strings (~200 B/row) before pivot_table ever runs. 
- That's >100 GB. full_Z and consensus_cluster are megabytes by comparison.



In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

In [ ]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

In [ ]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [ ]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

### Open primary cites from cbio

In [ ]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [ ]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

### Running prism

In [ ]:
verbose=True

res = prism.open_bayesprism(verbose=verbose)
print(len(res.genes))

In [ ]:
res.states

In [ ]:
res.theta_type.columns

In [ ]:
print(res.theta.shape)
res.theta.head(5)

In [ ]:
res.theta.tail(5)

### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [ ]:
res.cell_type_expression("Ductal cell type 1").shape

In [ ]:
res.cell_type_expression("Ductal cell type 1").head(3)

In [ ]:
res.cell_type_expression("Ductal cell type 2").shape

In [ ]:
res.cell_type_expression("Ductal cell type 2").head(3)

### 2. theta is now fixed -> expand Z to every gene

In [ ]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

verbose=True
force=False

df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        force=force, verbose=verbose)

force=False
verbose=True
fname = "count-matrix.txt"
fname_ad = fname.replace('.txt', '.h5ad')

adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

filename_ad = prism.root_singc / fname_ad
compression = "gzip"

verbose=True
fname_celltype = "all_celltype.txt"
adata = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

ref, s2t = prism.pseudobulk_reference(adata)

### ref, s2t

In [ ]:
print(ref.shape)
ref

In [ ]:
print(type(s2t), len(s2t))
s2t

### why Zfull resulted in 16550 genes?

Because full_Z reconstructs the full gene set, not the subset BayesPrism fitted on.

The three numbers you've seen trace it:

- 16550 — genes in full_Z, the whole expression matrix
- 1604 — genes in cell_type_expression, the marker-based fit
- ~10000 — after min_share/min_counts filtering

BayesPrism runs on gene_subset (marker/signature genes) for tractability and identifiability. That gave 1604. 

full_Z then projects the remaining ~15000 genes onto the fitted compartment basis — which is precisely why you built it: to recover the lncRNA/antisense loci (FAM83A-AS1, HOXA10-AS, HOXB-AS3/4, MIR7-3HG) that the marker fit excluded.

Shape is (153 samples, 10 cell types, 16550 genes) — build_ms_from_full_Z resolves that axis order automatically.

The consequence you should hold onto: 
- those ~15000 recovered genes are not Gibbs posterior estimates. 
- they're projections onto a basis fitted from 1604 genes, 
- so their sampling variance is structurally different 
  - no posterior shrinkage in the same sense, 
  - and their between-sample variation partly reflects the projection rather than compartment-specific evidence.

In [ ]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref)

In [ ]:
Zfull.shape

In [ ]:
dic = {}

for cell_state in res.states:
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z
    print(cell_state, Z.shape)


In [ ]:
i=0
key = list(dic.keys())[i]

print(key)
dic[key]

### Ductal 2 - malignant

In [ ]:
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")

In [ ]:
for g in ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "MIR7-3HG"]:
    if g in gfull:
        print(g, prism.gene_compartment_share(Zfull, gfull, res, g).head(3).round(3).to_dict())

In [ ]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]

prog2 = ["GATA6", "KRT17", "NEAT1", "H19", "DLEU1", "DLEU2"]

### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [ ]:
[g for g in prog1 if g in df_bulk.index]

### present in the scRNA reference?

In [ ]:
  
[g for g in prog1 if g in ref.columns]

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [ ]:
set(ref.index.to_list())

In [ ]:
set(s2t.index.to_list())

In [ ]:
adata.obs

In [ ]:
import re, numpy as np, pandas as pd

adata.obs["sample"] = adata.obs_names.to_series().str.extract(r"^([TN]\d+)_")[0].values
adata.obs["tissue"] = np.where(adata.obs["sample"].str.startswith("T"), "tumor", "normal")

print(adata.obs.groupby("tissue")["sample"].nunique())      # expect tumor 24, normal 11
print(pd.crosstab(adata.obs["cell_state"], adata.obs["tissue"]))

### Count Malignant Cells - accordingo to transcriptomics

In [ ]:
d2 = adata.obs["cell_state"].eq("Ductal cell type 2")
print(len(d2))
d2[:5]

In [ ]:
is_t = adata.obs["tissue"].eq("tumor")
print(np.sum(is_t))

In [ ]:
adata.obs["cell_state"] = np.where(d2 &  is_t, "Malignant ductal",
                          np.where(d2 & ~is_t, "Ductal cell type 2 normal",
                                   adata.obs["cell_state"]))
adata.obs

In [ ]:
from collections import Counter

Counter(adata.obs["cell_state"] )

In [ ]:
adata.obs["cell_type"]  = np.where(adata.obs["cell_state"].eq("Malignant ductal"),
                                   "malignant", adata.obs["cell_type"])

Counter(adata.obs["cell_type"] )

In [ ]:
ref2, s2t = prism.pseudobulk_reference(adata, state_key="cell_state", type_key="cell_type")
s2t.to_dict()

### LFC

calc_celltype_lfc() — each compartment vs the mean of the others, paired across samples by default. Paired is the right default here because every sample contributes every cell type, so pairing removes cohort/purity variance. This doubles as deconvolution QC: if the ductal compartment doesn't recover KRT19/TFF1/CEACAM6 and fibroblast doesn't recover COL1A1/POSTN, θ or the Peng reference is off and step 2 is meaningless.

### Critics

- Why not, for each cell type, tumor samples x normal samples
- Only Ductal 2 Tumor has no normal samples - to confirm


In [ ]:
res.__dict__.keys()

In [ ]:
res.states

### Prism programs

In [ ]:
Z_full, genes_full = prism.full_Z(res, df_bulk, ref)

### MalignantCluster

In [ ]:
from libs.prism_malig_lib import MalignantCluster

In [ ]:
type(res)

In [ ]:
cbio.root_mprog_disease

In [ ]:
root_mprog_cluster = cbio.root_mprog_disease / 'cluster'
mal_cell_name = "Ductal cell type 2"
kmax = 8
no_decouple = True
is_tahoe = True

'''
mc = pml.MalignantCluster(prism=prism, res=res, 
                          df_bulk=df_bulk, ref=ref, 
                          root_mprog_cluster = root_mprog_cluster,
                          mal_cell_name = mal_cell_name,
                          organ="Pancreas")
'''

import importlib, libs.prism_malig_lib as pml
importlib.reload(pml)
print(pml.__version__)

In [ ]:
mc = pml.MalignantCluster(prism=prism, res=res, 
                          df_bulk=df_bulk, ref=ref, 
                          root_mprog_cluster = root_mprog_cluster,
                          mal_cell_name = mal_cell_name,
                          organ="Pancreas")

mc

In [ ]:
X, diag = mc.prepare_malignant_matrix(decouple_purity=False, keep_genes=mc.program1_panel, drop_pattern=r"^N-")
print(X.shape)
X.head(3)

In [ ]:
X.tail(3)

In [ ]:
lista = [x for x in X.index if x.startswith('T-')]
X.shape[0], len(lista) == X.shape[0]

In [ ]:
diag.keys()

In [ ]:
diag["samples_excluded_by_filter"]

### pc_theta_pearson and pc_theta_pearson_raw

**The computation.** Run PCA on the samples × genes matrix, take the first 5 principal components, and correlate each PC's sample scores with `theta_mal` (each sample's malignant fraction). `pcs[:, i]` is one number per sample for PC *i*; `theta_mal.values` is one number per sample. `np.corrcoef(...)[0,1]` pulls the off-diagonal — the Pearson r between them.

You get 5 numbers, one per PC. Each answers: *does this dominant axis of variation track tumour purity?*

**The two versions:**

| | matrix | meaning |
|---|---|---|
| `pc_theta_pearson_raw` | `logx` — log2-CPM before decoupling | how much purity is in the data |
| `pc_theta_pearson` | `Xc` — the matrix you actually cluster | how much purity survives into the analysis |

With `decouple_purity=False` they're the same matrix, so the numbers match — your `[-0.596, -0.205, 0.227, -0.359, -0.158]` versus `[-0.596, -0.205, 0.228, -0.359, -0.159]`. The tiny differences are HVG selection, which happens between the two calls.

With `decouple_purity=True`, `Xc` holds residuals from regressing on `theta_mal`, and residuals are **orthogonal to their regressors by construction**. So `pc_theta_pearson` becomes ~1e-15 — pure floating-point noise. It proves the arithmetic worked, nothing about your data. That's why 0.20.1 added the `_raw` version and the `pc_theta_note`: I originally had you reading a number that can only ever be zero.

**Your actual numbers matter.** PC1 at r = −0.596 means ~36% of the leading component's variance is shared with purity, and PC4 at −0.359 adds more. The sign says low-purity samples score high on PC1. Since `X` is what produced the consensus clustering, the 134-gene tumour axis, and the 6/119 splits, purity is a live confound in all of them.

Which is the concrete reason to run `decouple_purity=True` and compare — with the standing caveat that basal-like PDAC is genuinely lower-purity, so some of that r is biology you'd be deleting.

In [ ]:
diag["pc_theta_pearson"]  # PC-vs-theta_mal

In [ ]:
diag["pc_theta_pearson_raw"]   # PC-vs-theta_mal on logx (pre-decoupling)

In [ ]:
diag["pc_theta_note"]          # warns the decoupled version is ~0 by construction

In [ ]:
diag["sample_mean_expr"]       # Xc.mean(axis=1) per sample

In [ ]:
diag["sample_total_Z"]         # ms.Z.sum(axis=1) per sample

In [ ]:
diag["theta_excluded"]

In [ ]:
diag["theta_kept"]

### Inspecting vars

In [ ]:
import inspect
print(pml.__version__)
print("drop_pattern" in inspect.signature(mc.prepare_malignant_matrix).parameters)

In [ ]:
info = mc.inspect_de_schema(genes=X.columns)
print(info.keys())
info["columns"]

In [ ]:
info["matches"]

In [ ]:
info["dtypes"]

In [ ]:
info["resolved_columns"]

In [ ]:
info["numeric_profile"]      # min / max / mean / frac_negative / n_unique


In [ ]:
info["signed_candidates"]

In [ ]:
cov = mc.index_coverage()
cov["n_lines_seen"], cov["n_lines_in_metadata"]

In [ ]:
cov["block_probe_counts"]        # min probes per block

In [ ]:
lista = cov["in_metadata_not_in_index"]
len(lista), lista[:5]

In [ ]:
cov["n_lines_seen"], cov["n_lines_in_metadata"]

That bears directly on the result you just got. X_fib2 has 9959 genes, so the large majority are recovered rather than fitted. If those 367 stroma-axis hits are concentrated among recovered genes, the signal may be a property of the projection, not the fibroblast compartment:

In [ ]:
cmap = {
    "malignant":  "Ductal cell type 2",
    "fibroblast": "Fibroblast cell",     # whatever Peng calls stellate/CAF
    "macrophage": "Macrophage cell",
    "endothelial": "Endothelial cell",
}

X_mal2 = mc.compartment_matrix(cmap["malignant"],  min_share=0.3, min_counts=10)
X_fib2 = mc.compartment_matrix(cmap["fibroblast"], min_share=0.3, min_counts=10)
print(X_mal2.shape, X_fib2.shape)

scores, cov = mc.program_scores(compartment_map=cmap, samples=X.index)   # tumours only

disc = mc.discretize_axes(scores)

print("in malignant", end=" ")
R_mal = mc.factorial_state_de(X_mal2, disc)
print("in fibroblast", end=" ")
R_fib = mc.factorial_state_de(X_fib2, disc)

print("-----------"*5)

fitted = set(res.cell_type_expression(cmap["fibroblast"]).index)   # the 1604
hits = R_fib.index[R_fib.filter(like="fdr_B_").iloc[:,0] < 0.05]
print(f"{len(set(hits) & fitted)}/{len(hits)} hits are fitted genes")
print(f"background: {len(fitted & set(R_fib.index))}/{len(R_fib)}")

print("-----------"*5)

common_genes = X_mal2.columns.intersection(X_fib2.columns)
Z_mal_common = ((X_mal2[common_genes] - X_mal2[common_genes].mean()) / X_mal2[common_genes].std())
Z_fib_common = ((X_fib2[common_genes] - X_fib2[common_genes].mean()) / X_fib2[common_genes].std())
Z_corr = pd.Series({gene: Z_mal_common[gene].corr(Z_fib_common[gene]) for gene in common_genes})

fitted = set(res.cell_type_expression(cmap["fibroblast"]).index)
print("fitted    genes, median r:", Z_corr[ Z_corr.index.isin(fitted)].median().round(3))
print("projected genes, median r:", Z_corr[~Z_corr.index.isin(fitted)].median().round(3))

### **The enrichment test is reassuring** and **The correlation test is the problem.**

**The enrichment test is reassuring.** 
- 109/367 hits are fitted genes (29.7%) 
  - against a background of 558/9959 (5.6%) 
  - a **5.3× enrichment**
  - not the depletion I warned about. 
  - so the 367-gene stromal signal concentrates in genes with genuine Gibbs posterior evidence, not in the projected majority. My concern there was wrong.

**The correlation test is the problem.** 
- I predicted projected genes near r=1 and fitted genes clearly lower. 
- Instead: fitted **0.927**, projected **0.905** — both extremely high, and fitted is *higher*.

**That means the malignant and fibroblast compartment matrices are ~92% correlated across samples, gene by gene, after z-scoring.**

Roughly 85% of the variance is shared. The deconvolution is separating compartment *means* (which is why marker programs score sensibly)
- but not compartment-specific *between-sample variation*. **Both matrices largely track the bulk.**

Note this is with `decouple=True` (the 0.31+ default), so it isn't θ — residualising on composition didn't remove it.

**Why your axis analyses survive this and the gene-level ones don't.** A program score is a contrast within a compartment: `basal − classical` subtracts the shared component, leaving the compartment-specific residual. That's why the axes behaved, why `couple_compartments` gave interpretable off-diagonal nulls, and why the `prolif`↔`iCAF` replication is credible. `factorial_state_de` on raw `X_mal2` vs `X_fib2` has no such cancellation — you're running DE on two near-copies of the same matrix.

So the 7-vs-367 contrast between compartments **cannot be read as compartment-specific biology**. Quantify the split directly:

If the specific fraction is ~0.04 for both classes, gene-level compartment claims aren't supportable at all and should be dropped from the writeup, keeping the contrast-based results.

This belongs in the summary as a limitation, and it's a genuine one: BayesPrism recovered compartment identity but not independent compartment-level sample variation. It also retroactively explains the zero off-diagonal in the factorial — with matrices this correlated, a "cross-compartment" test isn't really crossing anything.

### state feasibility

In [ ]:
X.iloc[:5, :10]

In [ ]:
dfs = mc.state_feasibility(n_samples=len(X), n_axes=3, levels=3)
dfs

### cmap

Two things needed: 

- marker sets, and a check on which compartments are even scoreable 
- θ for B cells and endothelium in PDAC bulk is often 1–3%, where Z is mostly prior.

In [ ]:
mc.df_theta.columns

In [ ]:
cmap = {
    "malignant":  "Ductal cell type 2",
    "fibroblast": "Fibroblast cell",     # whatever Peng calls stellate/CAF
    "macrophage": "Macrophage cell",
    "endothelial": "Endothelial cell",
    "immune":     "T cell",
    "humoral":    "B cell",
    "ductal":     "Ductal cell type 1",
    "acinar":     "Acinar cell",
}

rd = mc.compartment_readiness(cmap)
rd[["compartment","theta_median","n_genes_after_share","marker_coverage","verdict"]]

In [ ]:
cmap = {
    "malignant":  "Ductal cell type 2",
    "fibroblast": "Fibroblast cell",     # whatever Peng calls stellate/CAF
    "macrophage": "Macrophage cell",
    "endothelial": "Endothelial cell",
}

In [ ]:
scores, cov = mc.program_scores(compartment_map=cmap, samples=X.index)   # tumours only

In [ ]:
mat = [x for x in scores.index if x.startswith('T-')]
len(mat) == len(X), len(mat)

In [ ]:
scores

In [ ]:
cov

In [ ]:
R = mc.couple_compartments(scores)
print(R.shape)
R

In [ ]:
disc = mc.discretize_axes(scores)
disc

In [ ]:
dic_cross  = mc.axis_crosstab(disc)
dic_cross.keys()

In [ ]:
dic_cross['joint_label']

In [ ]:
dic_cross['counts']

In [ ]:
T = mc.state_signatures(X, mc.axis_crosstab(disc)["joint_label"], sort_by="fdr_nominal")
T

In [ ]:
# factorial decomposition — the better read for a 2x2
R = mc.factorial_state_de(X, disc)
R.head(3)

### no tumour–stroma interaction detected

Uniform p-values — that's a clean null, not a broken model. With 1971 genes, the smallest p under a true null is ~1/1971 ≈ 5e-4, and BH turns that into q ≈ 0.999. Exactly what you see. So: no tumour–stroma interaction detected.

The discriminating question is whether the main effects are non-null:

In [ ]:
R.fdr_interaction.describe()

In [ ]:
Rfdr = R[R.fdr_interaction < 0.1]
print(Rfdr.shape)
Rfdr

In [ ]:
print((R.filter(like="fdr_A_") < 0.05).sum())
print((R.filter(like="fdr_B_") < 0.05).sum())

In [ ]:
R.filter(like="fdr_").describe()

That's a much more interesting result than it looks, because of which matrix you ran it on.

X is the malignant compartment. So the two main effects mean different things:

A (134 genes) — the tumour axis predicts 134 malignant genes beyond its own defining markers, which were excluded. That's real validation: the basal/classical axis isn't noise. But it's a within-compartment test, so partly expected.
B (0 genes) — the stroma axis fails to predict malignant expression. That's not "the stroma axis is noise." It's already a crosstalk test, and it's null.

The cell you haven't tested is the interesting one: does the tumour axis predict fibroblast expression? Run the same model on the other compartment.

In [ ]:
coh = pd.Series(np.where(scores.index.str.contains("TCGA"), "TCGA", "CPTAC"), index=scores.index)
scores.groupby(coh)["fibroblast.axis_myCAF_minus_iCAF"].describe()

In [ ]:
from scipy.stats import levene, bartlett

col = "fibroblast.axis_myCAF_minus_iCAF"
a = scores.loc[coh == "TCGA",  col].dropna()
b = scores.loc[coh == "CPTAC", col].dropna()
print(len(a), len(b), a.std(), b.std())

levene(a, b), bartlett(a, b)

In [ ]:
for col in scores.columns:
    a = scores.loc[coh=="TCGA", col].dropna()
    b = scores.loc[coh=="CPTAC", col].dropna()
    print(f"{col:45s} sd {a.std():.3f}/{b.std():.3f}  levene p={levene(a,b).pvalue:.4f}")

In [ ]:
# and the tumour axis, for contrast
levene(scores.loc[coh=="TCGA","malignant.axis_basal_minus_classical"],
       scores.loc[coh=="CPTAC","malignant.axis_basal_minus_classical"])

### checking n_hvg filter in compartment_matrix

In [ ]:
# same expression floor, no variance selection, both compartments
X_mal2 = mc.compartment_matrix(cmap["malignant"],  min_share=0.3, min_counts=10)
X_fib2 = mc.compartment_matrix(cmap["fibroblast"], min_share=0.3, min_counts=10)
print(X_mal2.shape, X_fib2.shape)

In [ ]:
disc.head(3)

In [ ]:
R_mal = mc.factorial_state_de(X_mal2, disc)
R_fib = mc.factorial_state_de(X_fib2, disc)

In [ ]:
for nm, R in [("malignant", R_mal), ("fibroblast", R_fib)]:
    n = len(R)
    a = int((R.filter(like="fdr_A_") < 0.05).sum().iloc[0])
    b = int((R.filter(like="fdr_B_") < 0.05).sum().iloc[0])
    i = int((R.fdr_interaction < 0.05).sum())
    print(f"{nm:11s} n={n:5d}  A={a:4d} ({a/n:.2%})  B={b:4d} ({b/n:.2%})  int={i}")

In [ ]:
X_fib = mc.compartment_matrix(cmap["fibroblast"], min_share=0.3, min_counts=1)
R_fib = mc.factorial_state_de(X_fib, disc)
print((R_fib.filter(like="fdr_A_") < 0.05).sum())    # tumour -> stroma crosstalk
print((R_fib.filter(like="fdr_B_") < 0.05).sum())

In [ ]:
js = mc.joint_states(scores)
js["summary"]

In [ ]:
best = mc.choose_k(js["consensus"], min_cluster_frac=0.10)
best

In [ ]:
mc.state_profile(js)

In [ ]:
mc.axis_modality(scores)

In [ ]:
labels = js["consensus"][3]["labels"]
labels

In [ ]:
s2 = labels[labels==2].index
dffib = mc.df_theta.loc[s2, cmap["fibroblast"]]     # low fibroblast content?
dffib

In [ ]:
dffib.describe()

In [ ]:
small = labels[labels == 1].index
small

In [ ]:
set(s2) & set(small)

In [ ]:
s2 = labels[labels == 2].index
mc.df_theta.loc[s2, cmap["fibroblast"]].sort_values().round(4)

In [ ]:
scores, cov = mc.program_scores(compartment_map=cmap, samples=X.index)
cov[["compartment","program","n_found","r_with_theta"]]

In [ ]:
mc.axis_modality(scores)
js = mc.joint_states(scores)
mc.state_profile(js)

### Both cohorts and modality

The better use of two cohorts is replication, not deletion. Run the pipeline in each independently and keep what reproduces

An axis that is bimodal in both, or a basal↔myCAF coupling with the same sign in both, is far stronger evidence than anything from a merged analysis — and it doesn't require you to decide which cohort to trust.

If you want a discovery/validation split, use TCGA as discovery (n=68) and CPTAC as validation (n=44). Counterintuitive given TCGA is the noisier one, but discovery needs the power and validation needs the clean measurement.

If you do want to remove something, remove the specific samples rather than the cohort — the four with θ_fib < 0.05 from your sorted list, which min_theta=0.02 already partly handles. That's a stated QC criterion applied uniformly to both cohorts, which is defensible in a way that "we dropped TCGA" isn't.

One caveat on my own advice: TCGA-PAAD is known for low neoplastic cellularity in a substantial fraction of cases, so some of that extra dispersion is probably real data quality rather than batch. That argues for the QC threshold, not for dropping the cohort.


In [ ]:
for name in ["TCGA", "CPTAC"]:
    idx = scores.index[coh == name]
    sc_c, cov_c = mc.program_scores(compartment_map=cmap, samples=idx)
    print(name, len(idx))
    print("------"*5)
    print(mc.axis_modality(sc_c))
    print("------"*5)
    print(mc.couple_compartments(sc_c, n_perm=1000).head(5))
    print("------"*5)


### **Neither axis is bimodal in either cohort.**

This is the payoff. Two clean results, one negative and one positive.

TCGA: ΔBIC 7.4 and 9.6, both below the threshold, both flagged outlier-driven. CPTAC: ΔBIC −1.8 and −9.6, both unimodal. That's a consistent, replicated negative — **discrete subtypes are not supported in your data.** Your basal/classical and myCAF/iCAF axes are continua. The k=3 joint states, the 2×2 cross-tab, the 12-sample "iCAF group" — all were partitions of clouds. Stop pursuing a taxonomy and report continuous axes.

**One coupling replicates across cohorts:**

| | TCGA | CPTAC |
|---|---|---|
| `malignant.prolif` ↔ `fibroblast.iCAF` | r = −0.486, FDR 0.041 | r = −0.497, FDR 0.050 |

Same sign, near-identical magnitude, independently significant in both, on n=68 and n=44. TCGA's top hit (`prolif` ↔ `myCAF_minus_iCAF`, r = +0.446) is the same relationship read on the contrast. And this is already θ-controlled — `couple_compartments` defaults to `control_theta=True`.

That's your tumour–stroma crosstalk finding: **proliferative malignant programs co-occur with iCAF-depleted stroma.** It's exactly the non-cell-autonomous signal you asked about eight turns ago, and it survived the one test that matters — independent replication.

Nothing else replicates. TCGA's `emt`↔`macrophage.TAM` and CPTAC's `emt`↔`myCAF`, `M1`↔`endothelial.activated`, `SPP1`↔`tip_angio` each appear in one cohort only.

Two cautions on those non-replicating hits. Note the `n` column: 27–39 for macrophage and endothelial pairs, versus 44–68 for fibroblast. Those are the `min_theta` drops, and a correlation of 0.65 at n=27 has a 95% CI roughly 0.35–0.82 — wide enough that failure to replicate is uninformative either way. And `compartment_readiness` on those low-θ compartments is worth checking before you interpret them at all.

For the writeup, the honest framing is: two continuous compartment axes, cohort-stable for tumour and heteroscedastic for stroma, with a single replicated inverse coupling between malignant proliferation and inflammatory CAF content.

Three conditions, all required, in `axis_modality`:

```python
verdict = ("bimodal"             if d > 10 and w >= 0.15 and sep > 1.5 else
           "weak/outlier-driven" if d > 2 else
           "unimodal")
```

- **`delta_bic > 10`** — `BIC(1 component) − BIC(2 components)`. Positive favours two Gaussians; >10 is the conventional "strong evidence" threshold.
- **`min_component_weight >= 0.15`** — the smaller component must hold ≥15% of samples. Without this, a 2-component fit wins by devoting a tiny component to a few outliers.
- **`separation_sd > 1.5`** — distance between component means in pooled SD units. Two components can be statistically preferred yet overlap so heavily that no sample is confidently assigned.

Your TCGA axes failed on **`min_component_weight`**: 0.192 and 0.159 passed the 0.15 floor, `separation_sd` (1.53, 3.54) passed — so it was `delta_bic` that fell short (7.4 and 9.6, both under 10). Close to threshold, which is why "weak" rather than "unimodal."

CPTAC failed decisively on ΔBIC alone (−1.8, −9.6 — *negative*, i.e. one component is preferred outright), so the other conditions never mattered.

The thresholds are conventions, not laws. If you want to see how sensitive your verdicts are:

```python
mc.axis_modality(scores)[["axis","delta_bic","min_component_weight","separation_sd"]]
```

TCGA's 7.4 and 9.6 sit in the 2–10 "positive but not strong" band, so someone using ΔBIC > 6 would call them bimodal. But the negative ΔBIC in CPTAC is unambiguous, and that disagreement between cohorts is itself the answer — a genuinely bimodal axis should replicate. The skew values reinforce it: 1.56 and −1.01 indicate asymmetric tails, which is what makes a unimodal distribution look two-component to BIC.

### Delta-BIC

Delta BIC (ΔBIC) measures the difference between a candidate model's Bayesian information criterion BIC_{m} and the minimum BIC score {BIC}^{*} among a set of models. 

Calculated as Delta_BIC = {BIC}_m - {BIC}^*

it evaluates the relative evidence against a higher-scoring model. Lower BIC values indicate preferred, more parsimonious models.Interpreting Delta BIC Values0 to 2: Weak or bare difference; little to no evidence against the higher BIC model.2 to 6: Positive or moderate evidence favoring the model with the lower BIC.6 to 10: Strong evidence that the model with the lower BIC is superior.> 10: Very strong evidence against the higher BIC model, strongly supporting the minimum BIC choice.Watch this video for a high-level conceptual breakdown of how information criteria like AIC and BIC compare models and penalize complexity:17:25Multiple Regression, AIC, AICc, and BIC Basics59K views · 4 years agoYouTube · Brandon FoltzContext and UsageModel Selection: Used alongside maximum likelihood estimation to balance goodness of fit with model complexity while heavily penalizing extra parameters.Sample Size Sensitivity: Unlike AIC, BIC incorporates sample size (N) into its penalty term (\(K \ln(N)\)), making it more conservative as sample sizes increase.If you want to apply this, tell me:Are you comparing nested or non-nested models?What is your sample size and number of parameters?I can help you compute or interpret your specific values.

### **Summary**

Three corrections first — two of them matter a lot.

**"All statistical tests failed" is not what happened.** Several succeeded: 134 genes track the tumour axis at FDR<0.05 after excluding its defining markers; one tumour–stroma coupling replicated independently in both cohorts; the QC pipeline correctly identified normals, prior-dominated samples, and a compartment-specific batch effect. And a replicated *negative* on bimodality is a finding, not a failure.

**Point 2 claims a result you never computed.** You built `S` and `cond` from Tahoe, but `score_clusters_vs_tahoe` was never run on real data. Tahoe is unfinished, not negative. Writing it up as a failed test would be wrong.

**Point 3 conflates two different tests.** The Gaussian mixtures tested bimodality of the *axes*, not cohort distinctness. Cohort differences came from Levene/Bartlett, and they were **variance** differences with near-identical means (0.007 vs −0.005) — and only on some programs, not the tumour axis (p=0.77).

Here's the rewrite:

**Summary**

Bulk PDAC transcriptomics is dominated by stroma, so subtype signals are confounded with tumour cellularity. We deconvolved TCGA-PAAD and CPTAC3 with BayesPrism against a Peng 2019 reference, recovering compartment-specific expression for malignant, fibroblast, macrophage and endothelial compartments, and analysed the malignant compartment after within-compartment normalisation and purity decoupling.

Hard clustering of the malignant compartment proved unstable. Consensus clustering repeatedly returned degenerate partitions (6/119, 6/111, 3/108) in which the minority group had *both* basal and classical programs depressed — a signal-quality axis rather than a subtype axis. PAC selects for reproducibility, and isolating outliers is maximally reproducible, so the criterion favoured these splits. Sample-level QC (adjacent-normal samples, near-zero compartment θ) accounted for part but not all of it.

We therefore replaced discrete clustering with continuous per-compartment program scores. Two-component Gaussian mixtures rejected bimodality for both the tumour (basal−classical) and stromal (myCAF−iCAF) axes, independently in each cohort (CPTAC ΔBIC −1.8 and −9.6; TCGA 7.4 and 9.6, below the ΔBIC>10 threshold). **We find no support for discrete transcriptional subtypes in the deconvolved malignant compartment; both axes behave as continua.**

Cohort effects were heteroscedastic rather than locational: TCGA showed inflated variance across 13 of 15 programs (sign test p≈0.007), significantly so for the stromal axis (Levene p=0.031) but not the tumour axis (p=0.77). This dispersion, not a mean shift, explained a putative "iCAF-high" group that was 12/12 TCGA.

One cross-compartment coupling replicated: malignant proliferation correlated inversely with fibroblast iCAF content in TCGA (r=−0.486, FDR 0.041, n=68) and CPTAC (r=−0.497, FDR 0.050, n=44), after controlling for tumour purity. No cross-compartment differential expression was detectable in either direction, and no tumour×stroma interaction survived correction — though power analysis shows only interactions above ~1.4 log2 were detectable at these cell counts.

Two things I'd add before you finalise. State somewhere that Tahoe-100M connectivity scoring is set up but not yet run, so nobody reads its absence as a negative. And the whole analysis rests on one scRNA-seq reference — a Peng-specific misspecification would propagate to every compartment simultaneously, which no internal control here can detect.

In [ ]:
print(pml.__version__)